<center>
    <h2 style="font-size: 48px; margin-bottom: 5px;"><b>PROJECT QUIZ </b></h2>
</center>

<center>
    <h3 style="font-size: 36px; margin-top: 0px; margin-bottom: 5px;">- DELIVERABLE  -</h3>
</center>

<br> <br>

<center> <img src="https://www.alsacetech.org/wp-content/uploads/2017/05/cesi_CMJN-e1673249263164.png" alt="logo" width="200"></center>

<br> <br>

## Group 2 :
- ACACHA Elie
- PALE Kharice
- DEMIRHINDI Sude
- MERKICHE Isak
- LOUACHE Manyl

<br> <br>


# **CONTEXT**

Since the 1990s reducing energy consumption and greenhouse gas emissions have become a global objective. Transportation and delivery services contribute to these emissions improving delivery routes can help reduce the environmental impact. In this context ADEME (the French Environment and Energy Management Agency) promotes the development of mobility solutions for people and goods. CesiCDP wants to respond to this initiative by developing a method for managing delivery routes. Optimizing delivery routes can help companies reduce fuel consumption, travel time and operating costs.

# **PROBLEM**

Here we want to develop a method or a way to manage delivering route. Our main goal is to optimize those routes to reduces the company fuel comsumption, travel time and operating costs.The main problem of this project is to optimize the delivery routes on a road network. The objective is to find the efficient delivery routes for a convoy of delivering trucks that go to different customers and return to the starting point while minimizing travel time and respecting each delivery time window. This problem is known as the Capacitated Vehicle Routing Problem with Time Window (CVRPTW). Each truck has a capacity, and every customer must receive their deliveries in their time window and at each customer we must verify a certain service time duration. The algorithm must decide which customers each truck should visit and in which order. The main challenges are the number of possible deliveries routes the capacity limits of the trucks and the differences in traffic conditions and the differents time constraints. As the number of customers increases finding the best delivery routes becomes more difficult. This problem will the requires efficient algorithms.

# **OVERALL OBJECTIVES**
The main objective is to develop an algorithm of finding efficient delivery routes for realistic situations that involve several thousand customers.
The project has specific objectives:
- Model the delivery problem mathematically.
-	Develop an algorithm in Python to calculate delivery routes.
-	Respect the capacity limits of the trucks.
-	Minimize the travel time.
-	Evaluate the algorithms performance in terms of computation time and solution quality.
-	Test the algorithm on instances and compare its results with known reference solutions.
-	Time windows: each customer must be served during a specific period

# **Concrete Examples and Use Cases**

A typical example is a delivery company that must deliver packages to 100 customers using trucks. Each truck has a carrying capacity and every customer must be visited. A simple solution would be to visit customers in the order in which their addresses are received. However, this could create delivery routes, unnecessary travel and higher fuel consumption. With delivery route optimization the algorithm assigns customers to trucks and determines the order of visits to reduce the total travel time while respecting vehicle capacities. Another example is supermarket delivery service. Some customers may only be available to receive their orders between 10 a.m. And noon. In the version the algorithm must organize the delivery routes so that deliveries take place within these time windows. These applications show how delivery route optimization can improve delivery efficiency, reduce operating costs and support sustainable transportation.


# **Modeling and solution methods**


## Data Formalization

### 1. Benchmark instances and file formats

We test our methods on the **Solomon benchmark** (100 customers) and its extension by **Gehring & Homberger** (200 to 1000 customers), the standard benchmarks for the CVRPTW. All files come from the [PyVRP/VRPLIB](https://github.com/PyVRP/VRPLIB) repository. We read them with the Python package `vrplib`, which returns every instance as the same Python dictionary, whatever its file format.

The instances come in two file formats.

#### 1.1 Solomon format (`.txt`)

This is the original format of the benchmark. The file has a short header with the fleet, followed by a table with one row per node. Row `0` is the depot.

```
C101

VEHICLE
NUMBER     CAPACITY
  25         200

CUSTOMER
CUST NO.  XCOORD.   YCOORD.    DEMAND   READY TIME  DUE DATE   SERVICE   TIME
    0      40         50          0          0       1236          0
    1      45         68         10        912        967         90
    2      45         70         30        825        870         90
```

| Column | Meaning |
|---|---|
| `NUMBER` | number of available vehicles |
| `CAPACITY` | capacity of each vehicle (all vehicles are identical) |
| `CUST NO.` | node index (0 = depot) |
| `XCOORD.`, `YCOORD.` | coordinates of the node |
| `DEMAND` | quantity to deliver to the customer |
| `READY TIME`, `DUE DATE` | time window: earliest and latest start of service |
| `SERVICE TIME` | duration of the service at the customer |

The file contains **no distances**: `vrplib` computes them from the coordinates as real-valued Euclidean distances. It is read with `vrplib.read_instance(path, instance_format="solomon")`.

#### 1.2 VRPLIB / LKH-3 format (`.vrptw`)

This format follows the TSPLIB convention: a header of `KEY : value` lines, then one section per type of data.

```
NAME : C102
TYPE : CVRPTW
DIMENSION : 101          <- number of nodes (depot + 100 customers)
VEHICLES : 10
CAPACITY : 200
SERVICE_TIME : 90        <- one value for every customer
EDGE_WEIGHT_TYPE : EXACT_2D
NODE_COORD_SECTION       <- "node x y"
1 40 50
2 45 68
...
DEMAND_SECTION           <- "node demand"
TIME_WINDOW_SECTION      <- "node ready due"
DEPOT_SECTION            <- depot index, ended by -1
1
-1
EOF
```

Differences with the Solomon format:

- Nodes are numbered from **1** in the file; `vrplib` shifts them so that the depot is node **0**, as in the Solomon files.
- The service time is a **single value** for all customers. The depot has no service, so we set $s_0 = 0$.
- `EDGE_WEIGHT_TYPE` tells how distances are computed. With `EXACT_2D` and `EUC_2D`, `vrplib` returns the real Euclidean distance; with `FLOOR_2D`, it rounds the distance down to an integer. All the instances we use give real distances.
- In the LKH-3 files, the fleet is much tighter. In the file name `C102.100.10`, 100 is the number of customers and 10 the number of vehicles. Here $|K| = \lceil \sum_i d_i / Q \rceil$, so no vehicle is spare. The `.vrp` file of RC101 uses the same format but keeps the original Solomon fleet of 25 vehicles.

It is read with `vrplib.read_instance(path)`.

#### 1.3 Data actually used in our computations

Some fields of the files describe the instance but do not take part in the computations (`NAME`, `TYPE`, `COMMENT`, `DIMENSION`, `DEPOT_SECTION`, `SCALE`). The table below lists the fields that do take part, and how `vrplib` gives them to us.

| Data | Solomon `.txt` | `.vrptw` | `vrplib` key | Python shape | Symbol |
|---|---|---|---|---|---|
| Number of vehicles | `NUMBER` | `VEHICLES` | `vehicles` | integer | $\lvert K\rvert$ |
| Vehicle capacity | `CAPACITY` | `CAPACITY` | `capacity` | integer | $Q$ |
| Coordinates | `XCOORD.`, `YCOORD.` | `NODE_COORD_SECTION` | `node_coord` | $(n+1)\times 2$ | $(x_i, y_i)$ |
| Demands | `DEMAND` | `DEMAND_SECTION` | `demand` | $n+1$ | $d_i$ |
| Time windows | `READY TIME`, `DUE DATE` | `TIME_WINDOW_SECTION` | `time_window` | $(n+1)\times 2$ | $[b_i, e_i]$ |
| Service times | `SERVICE TIME` | `SERVICE_TIME` | `service_time` | $n+1$ or scalar | $s_i$ |
| Travel times | computed | computed | `edge_weight` | $(n+1)\times(n+1)$ | $\tau_{ij}$ |

The depot row gives $d_0 = 0$, and its time window $[b_0, e_0]$ is the working day: vehicles leave the depot at $b_0$ at the earliest and must be back by $e_0$.

#### 1.4 Instances used in this project

| Instance | Format | Customers $n$ | Vehicles $\lvert K\rvert$ | $Q$ | Total demand | Min. vehicles $\lceil \sum d_i / Q\rceil$ | $[b_0, e_0]$ | Used for |
|---|---|---|---|---|---|---|---|---|
| C101 | Solomon `.txt` | 100 | 25 | 200 | 1810 | 10 | [0, 1236] | ILP and ALNS, reference solution `C101.sol` |
| RC101 (first 20 customers) | `.vrp` (EUC_2D) | 20 | 25 | 200 | 430 | 3 | [0, 240] | ILP, tighter time windows |
| C102.100.10 | `.vrptw` (EXACT_2D) | 100 | 10 | 200 | 1810 | 10 | [0, 1236] | ALNS, wider time windows |
| C1_2_1 | Homberger `.txt` | 200 | 50 | 200 | 3530 | 18 | [0, 1351] | ALNS, reference solution `C1_2_1.sol` |

For the ILP, an instance can be cut down to its depot and its **first $n$ customers**, to study how the solving time grows with $n$.

For the reference instances, the `.sol` files give the best known solution: one line per route, then its cost. The published cost of C101 (827.3) uses distances truncated to one decimal, the original Solomon convention. With exact distances, the same routes cost **828.94**, so we always recompute the reference cost with our own distances before we compare.

### 2. Graph model

An instance is modelled as a **complete directed graph**

$$
G = (V, A), \qquad V = \{0\} \cup N, \qquad N = \{1, \dots, n\}, \qquad A = \{(i,j) \in V \times V : i \neq j\}
$$

- Node $0$ is the depot, where every route starts and ends.
- $N$ is the set of the $n$ customers.
- An arc $(i,j)$ is a direct trip from $i$ to $j$. Every pair of nodes is linked, so $\lvert A\rvert = n(n+1)$, which gives 650 arcs for 25 customers and 10 100 for 100 customers.

**Weight of an arc.** Following the Solomon convention, one unit of distance takes one unit of time, so the travel time is the Euclidean distance between the two nodes:

$$
\tau_{ij} = c_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}
$$

The travel time $\tau_{ij}$ is also the cost $c_{ij}$ that we minimise. The matrix is symmetric ($\tau_{ij} = \tau_{ji}$) and satisfies the triangle inequality, so going through an extra customer never shortens a trip.

**Weight of a node.** Each customer $i \in N$ carries a demand $d_i$, a time window $[b_i, e_i]$ and a service time $s_i$.

**Real-valued times.** Both methods use the exact, real Euclidean distances. The ILP uses the matrix `edge_weight` from `vrplib`, and the ALNS recomputes the same distances from the coordinates. In the ILP, the service start times are continuous variables, so travel times do not need to be integers.

### 3. Fleet

The fleet $K$ is **homogeneous**: every vehicle has the same capacity $Q = 200$ in all our instances. $\lvert K\rvert$ is an **upper bound**: a solution may use fewer vehicles. Each vehicle used performs a single route $R_k = (0, i_1, \dots, i_r, 0)$, and its load must not exceed the capacity:

$$
\sum_{i \in R_k} d_i \le Q
$$

Since demands cannot be split, at least $\lceil \sum_{i \in N} d_i / Q \rceil$ vehicles are needed. For C101 this gives 10 vehicles, and the best known solution indeed uses 10 routes out of the 25 available.

### 4. Time along a route

A vehicle leaves the depot at $T_0 = b_0$. When it goes from $i$ to $j$, it arrives at $a_j = T_i + s_i + \tau_{ij}$. If it arrives before the window opens, it waits, so service starts at

$$
T_j = \max(b_j, a_j)
$$

The route is feasible if $T_j \le e_j$ for every customer visited and if the vehicle is back at the depot by $e_0$. In C101 every customer needs $s_i = 90$ time units of service, which is long compared with the travel times (for example $\tau_{01} \approx 18.7$). Time windows, more than distances, therefore drive the number of customers a vehicle can serve.


## Complexity Proof
## 1. Complexity classification

The **Capacitated Vehicle Routing Problem with Time Windows (CVRPTW)** is an NP-hard combinatorial problem.

The objective is to minimize the total routing time cost while satisfying vehicle capacity constraints and customer time-window constraints. Every customer must be visited exactly once, and every vehicle route must start and end at the depot.

To establish its complexity, we consider the associated decision problem.

**CVRPTW-Decision**: Given a CVRPTW instance and a cost bound $B$, does there exist a set of feasible vehicle routes whose total cost is at most $B$ ?
The complexity classification is:
- Decision version: **NP-complete**.
- Optimization version: **NP-hard**.

We establish this classification by proving that the decision version belongs to NP and is NP-hard through a polynomial-time reduction from the decision version of the traveling salesman problem (TSP).

## 2. Proof that CVRPTW-Decision belongs to NP

To prove that CVRPTW-Decision belongs to NP, we must show that a proposed solution can be verified in polynomial time.

### Certificate

A certificate is a set of vehicle routes representing a solution structure for our problem:

$$
S=(R_1,R_2,\ldots,R_m)
\qquad m\leq |K|
$$

where each route starts and ends at the depot and gives the sequence of customers visited by its vehicle. The certificate has polynomial size: it lists each customer once and the depot twice per route, so at most $n + 2\lvert K\rvert$ nodes.

### Verification

Given a certificate $S$, we must verify the following conditions based on the problem constraints.

1. **Customer coverage:** Every customer is visited exactly once.
2. **Route structure:** Every vehicle route starts and ends at the depot, and the number of routes does not exceed the available fleet size.
3. **Capacity constraints:** For each route $R_k$, the total demand of its customers does not exceed the vehicle capacity $Q$.
4. **Time-window constraints:** Each customer is served within its time window.
5. **Cost bound:** the total cost of all routes is at most $B$.
6. **Depot return deadline:** every route returns by time $e_0$.

For time-window verification, let customer $i$ have a time window $[b_i,e_i]$, service duration $s_i$, and let $\tau_{ij}$ be the travel time from customer $i$ to customer $j$.

The earliest feasible service-start time at customer $j$, following customer $i$, is:

$$
T_j=\max\left(b_j,T_i+s_i+\tau_{ij}\right)
$$

We then verify that $T_j\leq e_j$.
If this inequality fails for any customer, the proposed route is infeasible. Otherwise, we continue along the route, and finally check that the vehicle is back at the depot by $e_0$.

Each node of the certificate is processed once, in constant time: one look-up in the travel-time matrix, one update of the load and of the time, and one mark in a boolean array to detect a customer visited twice or never. The whole verification therefore runs in $O(n + \lvert K\rvert)$, which is polynomial in the input size.

Therefore,

$$
\boxed{\text{CVRPTW-Decision}\in NP}
$$

## 3. Proof that CVRPTW-Decision is NP-Hard

We are going to prove this by reducing the decision problem of the travelling salesman problem to CVRPTW-Decision.

### 3.1. Source problem: TSP-Decision

An instance of TSP-Decision consists of a complete weighted graph $G=(V,E)$
a depot vertex $r\in V$, edge costs $c_{ij}$ and a cost bound $B$.

The question is whether there exists a Hamiltonian cycle that starts at $r$, visits every vertex exactly once, returns to $r$, and has total time cost at most $B$.

TSP-Decision is NP-complete (Karp, 1972, through the Hamiltonian cycle problem).

### 3.2. Polynomial-time transformation

Let $m=|V|$. Given an arbitrary TSP-Decision instance, construct a CVRPTW instance as follows:

- **Depot:** use the TSP vertex $r$ as the depot, relabeled $0$.
- **Customers:** each vertex in $V\setminus\{r\}$ becomes a customer.
- **Fleet size:** provide exactly **one** vehicle.
- **Demand:** set $d_i=1$ for every customer.
- **Vehicle capacity:** set $Q=m-1$, sufficient to serve all customers.
- **Travel costs:** preserve every TSP arc cost, $c'_{ij}=c_{ij}$.
- **Travel times:** set $\tau'_{ij}=c_{ij}$ so that total travel time equals TSP tour cost.
- **Service times:** set $s_i=0$ for every customer and for the depot.
- **Time windows:** assign $[0,H]$ to every customer and to the depot, with

$$
H=mC_{\max}+1,\qquad C_{\max}=\max_{(i,j)\in E}c_{ij}.
$$

A Hamiltonian cycle has exactly $m$ arcs, each costing at most $C_{\max}$; therefore its total travel time is at most $mC_{\max}<H$. These wide windows do not remove any TSP tour.

- **Objective bound:** keep the same bound $B$.

The constructed instance contains a polynomial number of nodes and arcs. Because the integer $H$ has polynomially many bits relative to the input encoding, this transformation runs in polynomial time.

### 3.3. Proof of reduction

We prove equivalence in both directions.

**Direction 1: TSP-Decision has a solution $\Rightarrow$ CVRPTW-Decision has a solution.**

Suppose that the TSP instance has a Hamiltonian cycle of total cost at most $B$. Use this cycle as the route of the single vehicle. Every customer is visited exactly once; capacity $Q=m-1$ allows all unit demands; service durations are zero; and the time horizon is large enough to accommodate the complete route. Since travel times equal the original TSP costs, the total CVRPTW travel time is also at most $B$. Therefore, the constructed CVRPTW instance is a **yes-instance**.

**Direction 2: CVRPTW-Decision has a solution $\Rightarrow$ TSP-Decision has a solution.**

Suppose that the constructed CVRPTW instance admits a feasible solution of total travel time at most $B$. There is only one vehicle, and the routing rules require it to start from the depot, serve every customer exactly once and return to the depot. Consequently, its route is a Hamiltonian cycle through all vertices of the original TSP graph. Because the travel times were copied from TSP arc costs, the original cycle has cost at most $B$. Therefore, the TSP instance is also a **yes-instance**.

We have established:

$$
\boxed{\text{TSP-Decision is a yes-instance}\iff\text{CVRPTW-Decision is a yes-instance}.}
$$

Hence:

$$
\text{TSP-Decision}\leq_p\text{CVRPTW-Decision}.
$$

Since TSP-Decision is NP-complete, CVRPTW-Decision is NP-hard. Combined with membership in NP, this yields:

$$
\boxed{\text{CVRPTW-Decision is NP-complete}.}
$$

Consequently, the **optimization version of the general CVRPTW is NP-hard**.


## Mathematical Modeling (ILP)

We use the compact **two-index formulation**. Since all vehicles are identical, one set of arc variables is enough for the whole fleet: we do not need to know *which* vehicle drives an arc, only *whether* an arc is used. Loads and times are tracked with Miller–Tucker–Zemlin (MTZ) style constraints, linearised with big-M.

### 1. Data

We use the notation of the Data Formalization: the graph $G = (V, A)$ with $V = \{0\} \cup N$, the travel times $\tau_{ij} = c_{ij}$, the demands $d_i$, the service times $s_i$ (with $s_0 = 0$), the time windows $[b_i, e_i]$, the depot window $[b_0, e_0]$, the capacity $Q$ and the fleet size $\lvert K\rvert$.

### 2. Variables

| Variable | Domain | Meaning |
|---|---|---|
| $x_{ij}$ | $\{0,1\}$, $(i,j) \in A$ | $=1$ if a vehicle travels directly from $i$ to $j$ |
| $T_i$ | $\mathbb{R}_{\ge 0}$, $i \in V$ | start of service at node $i$ ($T_0$ = departure time from the depot) |
| $u_i$ | $\mathbb{R}$, $i \in N$ | load of the vehicle just after it serves customer $i$ |

### 3. Objective

$$
\min \sum_{(i,j) \in A} c_{ij}\, x_{ij}
$$

### 4. Constraints

$$
\begin{aligned}
&(1)\ \textstyle\sum_{j : (i,j) \in A} x_{ij} = 1 && \forall i \in N && \text{each customer is left exactly once}\\
&(2)\ \textstyle\sum_{j : (j,i) \in A} x_{ji} = 1 && \forall i \in N && \text{each customer is reached exactly once}\\
&(3)\ \textstyle\sum_{j \in N} x_{0j} \le \lvert K\rvert && && \text{fleet size}\\
&(3')\ \textstyle\sum_{j \in N} x_{0j} \ge \left\lceil \sum_{i \in N} d_i / Q \right\rceil && && \text{valid inequality}\\
&(4)\ u_j \ge u_i + d_j - Q\,(1 - x_{ij}) && \forall (i,j) \in A,\ i, j \in N && \text{load propagation}\\
&(5)\ d_i \le u_i \le Q && \forall i \in N && \text{capacity}\\
&(6)\ T_j \ge T_i + s_i + \tau_{ij} - M_{ij}\,(1 - x_{ij}) && \forall (i,j) \in A,\ j \in N && \text{schedule consistency}\\
&(7)\ T_i + s_i + \tau_{i0} \le e_0 + M_{i0}\,(1 - x_{i0}) && \forall i \in N && \text{return to the depot before closing}\\
&(8)\ b_i \le T_i \le e_i && \forall i \in V && \text{time windows}
\end{aligned}
$$

Constraints (1) and (2) also balance the depot: $n$ arcs leave the customers and $n$ arcs reach them, so as many vehicles come back to the depot as leave it. All vehicles share the departure time $T_0$. This loses nothing: waiting is allowed, so leaving at $b_0$ is always possible.

### 5. Choice of the big-M values

When $x_{ij} = 1$, the term $M(1 - x_{ij})$ is zero and the constraint applies. When $x_{ij} = 0$, the constraint must hold for **every** feasible value of the variables, so that it is inactive. This gives the **smallest valid M**:

- **Time (6):** we need $T_i + s_i + \tau_{ij} - M_{ij} \le T_j$ for all $T_i \le e_i$ and $T_j \ge b_j$, so
$$M_{ij} = \max(0,\ e_i + s_i + \tau_{ij} - b_j).$$
- **Return (7):** in the same way, $M_{i0} = \max(0,\ e_i + s_i + \tau_{i0} - e_0)$.
- **Load (4):** we need $u_i + d_j - M \le u_j$ for all $u_i \le Q$ and $u_j \ge d_j$, so $M = Q$ is enough.

Any larger M gives the same integer solutions, but a weaker **linear relaxation**: a small fractional value of $x_{ij}$ is then enough to switch the constraint almost off. The lower bound gets worse, and the branch and bound must explore more nodes before it proves optimality.

### 6. Subtour elimination

Constraints (4) and (6) also remove **subtours**, that is, cycles of customers that do not go through the depot. Along such a cycle, the load would increase strictly at every arc ($d_j > 0$) and so would the time ($s_i + \tau_{ij} > 0$), which is impossible on a cycle. This argument needs positive demands or positive times, which holds in every Solomon instance ($d_i > 0$ and $s_i > 0$). The depot is left out of (4) and handled by (7), so cycles through the depot, which are the actual routes, stay allowed.

### 7. Valid inequality (3')

At least $\lceil \sum_i d_i / Q \rceil$ vehicles are needed to carry the total demand, so (3') removes no integer solution. It does tighten the linear relaxation, though. Without it, when the objective is the number of vehicles, CBC quickly finds a solution with the minimum fleet but struggles to **prove** that no smaller fleet exists.

### 8. Preprocessing

Before building the model, we remove the arcs between two customers that no feasible solution can use:

- **Time:** even when service at $i$ starts as early as possible, the vehicle reaches $j$ too late, that is $b_i + s_i + \tau_{ij} > e_j$.
- **Capacity:** the two demands together exceed the capacity, that is $d_i + d_j > Q$.

The arcs to and from the depot are always kept. The model gets smaller and the big-M values tighter. On C101 (100 customers), this leaves 4 512 arc variables out of 10 100, and 9 026 constraints.

### 9. Size of the model

The model has at most $n(n+1)$ binary variables, $2n + 1$ continuous variables and $O(n^2)$ constraints, so its size grows only quadratically. The difficulty lies elsewhere: the number of nodes explored by the branch and bound can grow exponentially with $n$, because the problem is NP-hard.


## Solution

### 1. Uses of the exact approach

The ILP above is solved with the branch and bound of CBC, through PuLP. Because the CVRPTW is NP-hard, it cannot handle the instances of several thousand customers targeted by the project. It is still useful in four ways.

- **Direct solution of small instances.** When the solver finishes, it returns a **proven optimal** solution. This works for small instances, and for larger ones when the time windows leave little choice. C101, with clustered customers and tight windows, is solved to optimality with its 100 customers. On harder instances, with random customers or wider windows, CBC solves this formulation comfortably only up to about 20 to 30 customers. Beyond that, the solving time explodes.
- **Benchmark for the heuristic.** On every instance the ILP solves, its optimum is the exact reference against which we measure the ALNS: $\text{gap} = 100\,(f_{ALNS} - f^*)/f^*$. On C101, the ALNS reaches the ILP optimum (828.94).
- **Lower bounds.** The **linear relaxation** of the model (with $x_{ij} \in [0,1]$) gives a lower bound on the optimum, $f_{LP} \le f^*$. When the solver stops at its time limit, it still returns its best bound. Either bound lets us say how far a heuristic solution is from the optimum *at most*, even when the optimum itself is unknown. The tighter the big-M values, the closer this bound is to $f^*$.
- **Upper bounds.** Every feasible solution found during the search, by the solver or by the ALNS, is an upper bound $f^* \le f$. A good ALNS solution could also be given to the solver as a starting point, so that it prunes more branches.

The same model also handles a **lexicographic objective**: first minimise the number of vehicles $\sum_j x_{0j}$, then minimise the distance subject to $\sum_j x_{0j} \le v^*$. This is how the Solomon literature ranks solutions.

For the large instances, we therefore need a method that finds good feasible solutions in a limited time, without any proof of optimality: a **metaheuristic**.

### 2. Choice of the metaheuristic: ALNS

We use the **Adaptive Large Neighbourhood Search** (ALNS) of Ropke & Pisinger (2006), which is among the most effective methods for vehicle routing problems with time windows.

**Why large moves.** A classic local search changes a solution only slightly, for example by moving one customer or swapping two. On the CVRPTW, most of these small moves are infeasible, because shifting one customer delays all the following ones and breaks their time windows. The search then gets stuck in a local optimum very quickly. The **Large Neighbourhood Search** (Shaw, 1998) makes much larger moves: it *destroys* part of the solution by removing 5 to 30 % of the customers, then *repairs* it by inserting them back. A single move can change both the assignment of customers to vehicles and the order of visits in several routes at once.

**Why adaptive.** There are many ways to destroy and to repair, and none is the best on every instance or at every stage of the search. ALNS keeps several operators and learns during the run which ones work best on the instance being solved.

**Why it fits our constraints.** The repair step inserts a customer only where capacity and time windows still hold. Every solution the algorithm visits is therefore feasible, and no penalty for violated constraints has to be tuned.

### 3. Definition of a solution

A solution $s$ is a set of routes plus a set of customers that are not served yet:

$$
s = \big(\{R_1, \dots, R_m\},\ U\big), \qquad R_k = (0, i_1, \dots, i_r, 0), \qquad m \le \lvert K\rvert
$$

Each customer of $N$ belongs to exactly one route or to $U$. Each route satisfies the capacity constraint $\sum_{i \in R_k} d_i \le Q$, the time windows and the depot deadline $e_0$. The routes are stored as Python lists of customers, without the depot.

**Objective function.**

$$
f(s) = \sum_{k=1}^{m} \sum_{(i,j) \in R_k} c_{ij} \;+\; \alpha\, m \;+\; \beta\, \lvert U\rvert
$$

- The first term is the total travel time, the quantity we minimise.
- $\alpha$ is a fixed cost per vehicle. It is $0$ by default; a large value (e.g. 1000) makes the search minimise the number of vehicles first.
- $\beta = 10^4$ penalises each customer that could not be inserted. It is larger than any possible saving in distance, so a solution that serves everyone always beats one that does not. The final solution has $U = \emptyset$.

**Initial solution.** We apply a repair operator (regret-2, defined below) to the empty solution $s_0 = (\emptyset, N)$. The same code builds the initial solution and repairs it during the search.

### 4. Feasibility of an insertion

Repair operators test thousands of insertions, so each test must be fast. For each route $(0, c_1, \dots, c_r, 0)$, we store two values at each stop $k$ (Savelsbergh, 1992):

- the **earliest start of service**, computed forwards: $T_k = \max\big(b_{c_k},\ T_{k-1} + s_{c_{k-1}} + \tau_{c_{k-1} c_k}\big)$;
- the **latest start of service** that keeps all later stops on time, computed backwards: $L_k = \min\big(e_{c_k},\ L_{k+1} - \tau_{c_k c_{k+1}} - s_{c_k}\big)$, with $L_{r+1} = e_0$.

Inserting customer $u$ between two consecutive stops $i$ and $j$ is feasible if and only if:

1. **capacity:** $\text{load}(R) + d_u \le Q$;
2. **$u$ is on time:** $T_u = \max(b_u,\ T_i + s_i + \tau_{iu}) \le e_u$;
3. **nobody after $u$ is late:** $\max(b_j,\ T_u + s_u + \tau_{uj}) \le L_j$.

Each test costs $O(1)$ per position, instead of re-simulating the whole route. The cost of the insertion is the detour $\Delta_{u,ij} = c_{iu} + c_{uj} - c_{ij}$.

### 5. Neighbourhood structure: destroy operators

A destroy operator removes $q$ customers from the current solution and puts them in $U$. At each iteration, $q$ is drawn at random between $5\,\%$ and $30\,\%$ of $n$, capped at 40, so the size of the moves varies. The neighbourhood of $s$ is the set of solutions obtained by one destroy step followed by one repair step.

| Operator | Customers removed | Role |
|---|---|---|
| **Random** | $q$ customers chosen uniformly | diversification |
| **Worst** | the customers whose removal saves the most distance, $c_{p u} + c_{u n} - c_{p n}$, where $p$ and $n$ are the stops before and after $u$ | moves badly placed customers |
| **Shaw (related)** | a random seed customer, then customers similar to those already removed | similar customers can be swapped between routes |
| **Route** | one whole route, short routes being more likely | lets the repair rebuild the solution with one vehicle fewer |

**Relatedness** (Shaw removal). Two customers are similar when they are close, open their windows at similar times and have similar demands:

$$
R(i,j) = 9\,\frac{c_{ij}}{\max c} + 3\,\frac{\lvert b_i - b_j\rvert}{e_0} + 2\,\frac{\lvert d_i - d_j\rvert}{\max d}
$$

A smaller value means more related. The weights 9, 3 and 2 come from Ropke & Pisinger (2006).

**Controlled randomness** (Worst and Shaw). The candidates are sorted from best to worst, and the one at index $\lfloor y^p \cdot \lvert L\rvert\rfloor$ is chosen, where $L$ is the sorted list of candidates and $y$ is uniform in $[0, 1)$. A large exponent $p$ almost always picks the first candidate, while $p = 1$ picks uniformly. We use $p = 3$ for Worst and $p = 6$ for Shaw.

### 6. Repair operators

All repair operators repeat the same step: pick a customer of $U$ and insert it at its cheapest feasible position. A new empty route is also offered as long as fewer than $\lvert K\rvert$ vehicles are used. The operators differ only in **which customer is inserted next**:

| Operator | Next customer inserted |
|---|---|
| **Greedy** | the customer with the cheapest insertion $\min_u c_{u,1}$, where $c_{u,h}$ is the cost of inserting $u$ in its $h$-th best route |
| **Greedy with noise** | the same, with a random noise of $\pm 2.5\,\%$ of the largest distance added to each cost |
| **Regret-2** | the customer with the largest regret $c_{u,2} - c_{u,1}$ |
| **Regret-3** | the customer with the largest regret $\sum_{h=2}^{3} (c_{u,h} - c_{u,1})$ |

The **regret** measures how much more a customer will cost if we do not insert it now in its best route. Regret operators therefore insert the "difficult" customers first, those with few good positions left, and keep the easy ones for the end. A customer that fits in only one route gets the highest priority. The noise in the greedy operator lets it build different solutions from the same partial solution.

### 7. Guidance mechanisms

**Adaptive operator selection.** Each destroy and repair operator $o$ has a weight $w_o$, initialised to 1. At each iteration, one destroy and one repair operator are drawn independently by **roulette wheel**:

$$
P(o) = \frac{w_o}{\sum_{o'} w_{o'}}
$$

The chosen pair receives a score $\psi$ depending on the outcome of the iteration:

| Outcome | Score $\psi$ |
|---|---|
| new best solution | $\sigma_1 = 33$ |
| better than the current solution | $\sigma_2 = 9$ |
| worse, but accepted | $\sigma_3 = 13$ |
| rejected | $0$ |

A worse but accepted solution scores more than an improving one ($\sigma_3 > \sigma_2$). This rewards operators that help the search leave a local optimum.

Every segment of $L = 100$ iterations, the weights are updated with a reaction factor $\rho = 0.1$:

$$
w_o \leftarrow \max\!\Big(0.05,\ (1 - \rho)\, w_o + \rho\, \frac{\pi_o}{\theta_o}\Big)
$$

Here $\pi_o$ is the total score of $o$ during the segment and $\theta_o$ the number of times it was used. Operators that produce good solutions are chosen more and more often. The floor of $0.05$ keeps every operator in play.

**Acceptance: simulated annealing.** A candidate $s'$ always replaces the current solution $s$ if $f(s') < f(s)$. Otherwise it is accepted with probability

$$
P(\text{accept}) = e^{-(f(s') - f(s)) / T}
$$

- The **initial temperature** $T_0$ is set so that a solution $5\,\%$ worse than the initial solution is accepted with probability $0.5$: $T_0 = -0.05\, f(s_0) / \ln 0.5$.
- The temperature then decreases geometrically, $T \leftarrow \gamma\, T$, with $\gamma$ chosen so that $T$ reaches $10^{-4}\, T_0$ at the last iteration.

At the start, many worse solutions are accepted, which favours exploration. At the end, almost only improvements are accepted, which favours intensification. The best solution ever found is stored separately and returned at the end.

**Stopping criterion.** A fixed number of iterations (2000 by default), with an optional time limit.

### 8. Parameters

| Parameter | Value | Role |
|---|---|---|
| iterations | 2000 | stopping criterion |
| $q$ | between 5 % and 30 % of $n$, at most 40 | size of the destroy step |
| $p_{\text{worst}}$, $p_{\text{Shaw}}$ | 3, 6 | randomness of Worst and Shaw removal |
| noise | 0.025 | amplitude of the noise in greedy with noise |
| $\sigma_1, \sigma_2, \sigma_3$ | 33, 9, 13 | operator scores |
| $\rho$, $L$ | 0.1, 100 | weight update |
| $T_0$ | 5 % worse accepted with probability 0.5 | initial temperature |
| $T_{\text{end}}$ | $10^{-4}\, T_0$ | final temperature |
| $\alpha$, $\beta$ | 0, $10^4$ | cost per vehicle, penalty per unserved customer |

Most values come from Ropke & Pisinger (2006). They are natural factors to vary in the experimental study.


# **Implementation**

## Pseudo-Code

The algorithm described in the Solution section is written below in pseudo-code. $f$ is the objective function, $D$ and $R$ are the sets of destroy and repair operators, and $\psi$ is the score of an iteration.

**Main loop**

```
ALNS(instance, N_iter):
    s      ← REPAIR_REGRET2( (∅, N) )             # initial solution: repair the empty solution
    s_best ← s
    T      ← -0.05 · f(s) / ln(0.5)               # a 5 % worse solution is accepted with prob. 0.5
    γ      ← (10^-4)^(1 / N_iter)                 # cooling rate: T reaches 10^-4 · T0 at the end
    w_o ← 1,  π_o ← 0,  θ_o ← 0   for every operator o ∈ D ∪ R

    for it = 1 .. N_iter:
        d ← ROULETTE(D, w)                         # P(o) = w_o / Σ w
        r ← ROULETTE(R, w)
        s' ← r( d(s) )                             # destroy q customers, then re-insert them

        if f(s') < f(s_best):                      # new best solution
            s_best ← s' ;  s ← s' ;  ψ ← σ1
        else if f(s') < f(s):                      # improves the current solution
            s ← s' ;  ψ ← σ2
        else if RANDOM() < exp( -(f(s') - f(s)) / T ):   # worse, accepted by simulated annealing
            s ← s' ;  ψ ← σ3
        else:
            ψ ← 0                                  # rejected

        π_d ← π_d + ψ ;  π_r ← π_r + ψ ;  θ_d ← θ_d + 1 ;  θ_r ← θ_r + 1
        T ← γ · T

        if it mod L = 0:                           # end of a segment: update the weights
            for every operator o used in the segment:
                w_o ← max(0.05, (1 - ρ) · w_o + ρ · π_o / θ_o)
            π ← 0 ;  θ ← 0

    return s_best
```

**Repair (shared by the four repair operators)**

```
REPAIR(s = (routes, U), k):                        # k = 1: greedy, k = 2 or 3: regret-k
    for every u ∈ U and every route R (plus one empty route if m < |K|):
        c[u, R] ← cheapest feasible insertion cost of u in R   (∞ if none)

    while U ≠ ∅ and some c[u, R] < ∞:
        if k = 1:  u* ← argmin_u  min_R c[u, R]
        else:      u* ← argmax_u  Σ_{h=2..k} ( c_{u,h} - c_{u,1} )   # c_{u,h}: h-th best route of u
        insert u* at its cheapest feasible position, in its best route R*
        U ← U \ {u*}
        recompute the column c[·, R*] only             # the other routes did not change

    return (routes, U)                             # customers left in U are penalised by β
```

**Insertion test of customer $u$ between consecutive stops $i$ and $j$**

```
FEASIBLE(R, u, i, j):
    if load(R) + d_u > Q:                         return false     # capacity
    T_u ← max(b_u, T_i + s_i + τ_iu)
    if T_u > e_u:                                 return false     # u is late
    if max(b_j, T_u + s_u + τ_uj) > L_j:          return false     # a later stop becomes late
    return true
```

## Different stages

The program runs in five stages.

**Stage 1: reading the instance.** `load_instance` reads the file with `vrplib` and turns it into numpy arrays: coordinates, demands $d_i$, time windows $[b_i, e_i]$, service times $s_i$ (with $s_0 = 0$), capacity $Q$ and fleet size $\lvert K\rvert$. It then precomputes the two $(n+1) \times (n+1)$ matrices used by the search:
- the exact Euclidean distances $c_{ij} = \tau_{ij}$;
- the relatedness $R(i,j)$ used by the Shaw removal.

Both cost $O(n^2)$ time and memory, once.

**Stage 2: data structures.**
- A **`Solution`** stores its routes as lists of customers, without the depot, and the list $U$ of unserved customers. It computes $f(s)$.
- A **`RouteData`** wraps one route with its cache: load, earliest starts $T_k$ (forward pass) and latest starts $L_k$ (backward pass). The cache is rebuilt in $O(r)$ after each change to the route. In return, `best_insertion` tests **all the positions of the route at once** with numpy vector operations, using the three conditions of the insertion test.

**Stage 3: initial solution.** Regret-2 repair applied to the empty solution: all customers start in $U$ and are inserted one by one.

**Stage 4: main loop.** Each iteration:
1. draws a destroy and a repair operator by roulette wheel;
2. destroys $q$ customers, then repairs. The repair dominates the cost of an iteration. Building the cost matrix tests every removed customer at every position, $O(q \cdot n)$ in total. Each insertion then recomputes only the column of the modified route;
3. evaluates $f(s')$ and applies the acceptance rule;
4. updates the scores, cools the temperature, and every $L = 100$ iterations updates the weights.

The search is **seeded** (`numpy.random.default_rng(seed)`), so a run can be reproduced exactly.

**Stage 5: checking and output.** `check_solution` recomputes the cost of the best solution **from scratch**, without reusing any code from the algorithm. It verifies that each customer is visited exactly once, the fleet size, the capacities, every time window and the return to the depot. If the ALNS had a bug in its fast insertion test, this independent check would detect it. The function `alns` returns the best solution and a history (current and best cost at each iteration, operator weights, runtime) used in the experimental study.

## Implementation

The code uses only `numpy`, `pandas`, `matplotlib` and `vrplib`. The instance files are in the `data/` folder of the project.


### Imports

In [1]:
import time                      # standard library, only used to measure runtimes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vrplib

DATA_DIR = "data"

### Stage 1: reading an instance

Node 0 is the depot and nodes $1..n$ are the customers. The relatedness matrix of the Shaw removal is precomputed here. `load_bks` reads a `.sol` file and recomputes its cost with our distances.

In [2]:
def load_instance(path):
    # Solomon files (.txt) need the "solomon" format; .vrptw / .vrp files use the default VRPLIB format
    fmt = "solomon" if path.lower().endswith(".txt") else "vrplib"
    raw = vrplib.read_instance(path, instance_format=fmt)

    n_nodes = len(raw["demand"])          # depot + customers

    # Service time can be given per node (Solomon) or as a single number (.vrptw): we always build an array
    service = np.broadcast_to(np.asarray(raw["service_time"], dtype=float), (n_nodes,)).copy()
    service[0] = 0.0                      # no service at the depot

    # Exact Euclidean distances recomputed from the coordinates (same convention for every file format).
    # Travel time = distance, as in the Solomon benchmark.
    coords = np.asarray(raw["node_coord"], dtype=float)
    dist = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2))

    tw = np.asarray(raw["time_window"], dtype=float)
    inst = {
        "name": raw["name"],
        "coords": coords,
        "demand": np.asarray(raw["demand"], dtype=float),
        "capacity": float(raw["capacity"]),
        "n_vehicles": int(raw["vehicles"]),   # fleet size limit
        "ready": tw[:, 0],                    # e_i: earliest service start
        "due": tw[:, 1],                      # l_i: latest service start
        "service": service,                   # s_i
        "dist": dist,                         # d_ij (= travel time t_ij)
        "n_customers": n_nodes - 1,
    }

    # Relatedness between two customers for the Shaw removal (smaller = more similar).
    # Weighted sum of normalised distance, difference of time-window opening and difference of demand.
    # Weights 9 / 3 / 2 come from Ropke & Pisinger (2006).
    d_norm = dist / dist.max()
    t_norm = np.abs(inst["ready"][:, None] - inst["ready"][None, :]) / max(1.0, inst["due"][0])
    q_norm = np.abs(inst["demand"][:, None] - inst["demand"][None, :]) / max(1.0, inst["demand"].max())
    inst["relatedness"] = 9 * d_norm + 3 * t_norm + 2 * q_norm
    return inst


def load_bks(sol_path, inst):
    # Read a .sol file and recompute its cost with OUR distances, so that gaps are measured consistently
    sol = vrplib.read_solution(sol_path)
    d = inst["dist"]
    cost = sum(d[[0] + r, r + [0]].sum() for r in sol["routes"])
    return cost, len(sol["routes"])


inst = load_instance(f"{DATA_DIR}/C101.txt")
print(inst["name"], "|", inst["n_customers"], "customers |", inst["n_vehicles"], "vehicles of capacity", inst["capacity"])
print("Known-best solution (recomputed cost, routes):", load_bks(f"{DATA_DIR}/C101.sol", inst))

C101 | 100 customers | 25 vehicles of capacity 200.0
Known-best solution (recomputed cost, routes): (np.float64(828.9368669428341), 10)


### Stage 2: data structures and fast insertion test

`RouteData.update` runs the forward and backward passes; `best_insertion` applies the three conditions of the insertion test to every position of the route at once.

In [3]:
class RouteData:
    # One route, with the cached information needed to test insertions in O(1) per position

    def __init__(self, inst, customers):
        self.customers = list(customers)   # customers only, without the depot
        self.update(inst)

    def update(self, inst):
        # Recompute the cache after the route changed (forward and backward passes)
        d, s, e, l = inst["dist"], inst["service"], inst["ready"], inst["due"]
        seq = np.array([0] + self.customers + [0])   # full sequence with the depot at both ends
        m = len(seq)
        start = np.empty(m)
        latest = np.empty(m)

        # Forward pass: earliest service start at each stop (waiting is allowed if we arrive early)
        start[0] = e[0]
        for k in range(1, m):
            arrival = start[k - 1] + s[seq[k - 1]] + d[seq[k - 1], seq[k]]
            start[k] = max(arrival, e[seq[k]])

        # Backward pass: latest service start at each stop that keeps the rest of the route feasible
        latest[-1] = l[0]                           # must be back at the depot before it closes
        for k in range(m - 2, -1, -1):
            latest[k] = min(l[seq[k]], latest[k + 1] - d[seq[k], seq[k + 1]] - s[seq[k]])

        self.seq, self.start, self.latest = seq, start, latest
        self.load = inst["demand"][seq].sum()

    def best_insertion(self, inst, u):
        # Return (extra distance, position) of the cheapest FEASIBLE insertion of customer u,
        # or (inf, -1) if u cannot be inserted anywhere in this route
        if self.load + inst["demand"][u] > inst["capacity"]:          # condition 1: capacity
            return np.inf, -1

        d, s, e, l = inst["dist"], inst["service"], inst["ready"], inst["due"]
        prev, nxt = self.seq[:-1], self.seq[1:]   # all (i, j) pairs of consecutive stops, vectorised

        start_u = np.maximum(self.start[:-1] + s[prev] + d[prev, u], e[u])
        arrival_next = start_u + s[u] + d[u, nxt]
        feasible = (start_u <= l[u]) & (np.maximum(arrival_next, e[nxt]) <= self.latest[1:])  # conditions 2 and 3
        if not feasible.any():
            return np.inf, -1

        extra = d[prev, u] + d[u, nxt] - d[prev, nxt]   # detour created by the insertion
        extra = np.where(feasible, extra, np.inf)
        pos = int(np.argmin(extra))
        return float(extra[pos]), pos


class Solution:
    # A solution = list of routes (lists of customers) + list of customers not served yet

    def __init__(self, routes, unassigned):
        self.routes = [list(r) for r in routes if len(r) > 0]   # empty routes are dropped
        self.unassigned = list(unassigned)

    def copy(self):
        return Solution(self.routes, self.unassigned)

    def distance(self, inst):
        d = inst["dist"]
        return sum(d[[0] + r, r + [0]].sum() for r in self.routes)

    def objective(self, inst, params):
        # Total distance + optional fixed cost per vehicle + big penalty per unserved customer
        return (self.distance(inst)
                + params["vehicle_cost"] * len(self.routes)
                + params["unassigned_penalty"] * len(self.unassigned))

### Destroy operators

In [4]:
def removal_size(inst, rng, params):
    # Number of customers to remove: random between q_min and q_max
    n = inst["n_customers"]
    q_min = max(2, int(params["q_min_frac"] * n))
    q_max = max(q_min, min(params["q_max_abs"], int(params["q_max_frac"] * n)))
    return int(rng.integers(q_min, q_max + 1))


def remove_customers(sol, to_remove):
    # Build a new solution where the customers in to_remove are taken out of their routes
    to_remove = set(to_remove)
    routes = [[c for c in r if c not in to_remove] for r in sol.routes]
    return Solution(routes, sol.unassigned + list(to_remove))


def random_removal(sol, inst, rng, params):
    customers = [c for r in sol.routes for c in r]
    q = min(removal_size(inst, rng, params), len(customers))
    return remove_customers(sol, rng.choice(customers, size=q, replace=False).tolist())


def worst_removal(sol, inst, rng, params):
    d = inst["dist"]
    q = removal_size(inst, rng, params)
    routes = [list(r) for r in sol.routes]
    removed = []
    while len(removed) < q:
        # Saving obtained by removing each customer: d(prev,c) + d(c,next) - d(prev,next)
        candidates, savings = [], []
        for r in routes:
            if not r:
                continue
            seq = np.array([0] + r + [0])
            g = d[seq[:-2], seq[1:-1]] + d[seq[1:-1], seq[2:]] - d[seq[:-2], seq[2:]]
            candidates.extend(r)
            savings.extend(g.tolist())
        if not candidates:
            break
        order = np.argsort(savings)[::-1]                              # biggest saving first
        chosen = candidates[order[int(rng.random() ** params["worst_p"] * len(order))]]
        removed.append(chosen)
        for r in routes:                                                # savings change after each removal,
            if chosen in r:                                             # so they are recomputed in the loop
                r.remove(chosen)
                break
    return Solution(routes, sol.unassigned + removed)


def shaw_removal(sol, inst, rng, params):
    R = inst["relatedness"]
    customers = [c for r in sol.routes for c in r]
    if not customers:
        return sol.copy()
    q = min(removal_size(inst, rng, params), len(customers))
    removed = [customers[rng.integers(len(customers))]]            # random seed customer
    remaining = set(customers) - set(removed)
    while len(removed) < q:
        ref = removed[rng.integers(len(removed))]                   # compare with a random removed customer
        rem = np.array(sorted(remaining))
        order = rem[np.argsort(R[ref, rem])]                        # most related first
        chosen = int(order[int(rng.random() ** params["shaw_p"] * len(order))])
        removed.append(chosen)
        remaining.discard(chosen)
    return remove_customers(sol, removed)


def route_removal(sol, inst, rng, params):
    if not sol.routes:
        return sol.copy()
    sizes = np.array([len(r) for r in sol.routes], dtype=float)
    prob = (1 / sizes) / (1 / sizes).sum()                          # small routes are more likely to be removed
    k = rng.choice(len(sol.routes), p=prob)
    return remove_customers(sol, sol.routes[k])

### Repair operators

The four repair operators share `insert_customers`; only the rule that picks the next customer changes (`k_regret`, `noise`).

In [5]:
def insert_customers(sol, inst, rng, params, k_regret=1, noise=False):
    routes = [RouteData(inst, r) for r in sol.routes]
    pending = list(sol.unassigned)
    rng.shuffle(pending)                                   # random tie-breaking
    noise_amp = params["noise"] * inst["dist"].max() if noise else 0.0

    def column(rd):
        # Cheapest insertion cost and position of every pending customer in route rd
        cost = np.empty(len(pending))
        pos = np.empty(len(pending), dtype=int)
        for i, u in enumerate(pending):
            cost[i], pos[i] = rd.best_insertion(inst, u)
        if noise_amp > 0:
            cost = np.maximum(0.0, cost + noise_amp * rng.uniform(-1, 1, len(pending)))
        return cost, pos

    def candidate_routes():
        # Existing routes + one empty route if another vehicle is still available
        return routes + ([RouteData(inst, [])] if len(routes) < inst["n_vehicles"] else [])

    cand = candidate_routes()
    cost = np.empty((len(pending), len(cand)))
    posm = np.empty((len(pending), len(cand)), dtype=int)
    for j, rd in enumerate(cand):
        cost[:, j], posm[:, j] = column(rd)
    cost[:, len(routes):] += params["vehicle_cost"]       # opening a new route has a fixed cost

    while pending:
        best = cost.min(axis=1)                            # best insertion cost of each pending customer
        if np.all(np.isinf(best)):
            break                                          # nobody can be inserted anymore

        # ---- choose which customer to insert ----
        if k_regret <= 1:
            i = int(np.argmin(best))                       # greedy
        else:
            big = 1e9                                      # infeasible routes count as very expensive
            sorted_cost = np.sort(np.where(np.isinf(cost), big, cost), axis=1)
            kk = min(k_regret, sorted_cost.shape[1])
            if kk > 1:
                regret = (sorted_cost[:, 1:kk] - sorted_cost[:, [0]]).sum(axis=1)
            else:
                regret = np.zeros(len(pending))
            regret = np.where(np.isinf(best), -np.inf, regret)   # never pick a customer that fits nowhere
            ties = np.flatnonzero(regret == regret.max())
            i = int(ties[np.argmin(best[ties])])           # tie-break on the cheapest insertion

        # ---- insert it in its best route at its best position ----
        j = int(np.argmin(cost[i]))
        rd = cand[j]
        rd.customers.insert(posm[i, j], pending[i])
        rd.update(inst)
        opened_new_route = j >= len(routes)
        if opened_new_route:
            routes.append(rd)

        pending.pop(i)
        cost = np.delete(cost, i, axis=0)
        posm = np.delete(posm, i, axis=0)
        if not pending:
            break

        # ---- update the cost matrix: only the modified route changed ----
        cost[:, j], posm[:, j] = column(rd)
        if opened_new_route:
            cost, posm = cost[:, :len(routes)], posm[:, :len(routes)]
            cand = candidate_routes()
            if len(cand) > len(routes):                    # offer a new empty route again
                c, p = column(cand[-1])
                cost = np.column_stack([cost, c + params["vehicle_cost"]])
                posm = np.column_stack([posm, p])

    return Solution([rd.customers for rd in routes], pending)


def greedy_repair(sol, inst, rng, params):
    return insert_customers(sol, inst, rng, params, k_regret=1)

def greedy_noise_repair(sol, inst, rng, params):
    return insert_customers(sol, inst, rng, params, k_regret=1, noise=True)

def regret2_repair(sol, inst, rng, params):
    return insert_customers(sol, inst, rng, params, k_regret=2)

def regret3_repair(sol, inst, rng, params):
    return insert_customers(sol, inst, rng, params, k_regret=3)


DESTROY = [random_removal, worst_removal, shaw_removal, route_removal]
REPAIR = [greedy_repair, greedy_noise_repair, regret2_repair, regret3_repair]

### Stage 5: independent solution checker

In [6]:
def check_solution(inst, routes):
    # Return (total distance, list of errors). An empty error list means the solution is feasible.
    d, s, e, l, q = inst["dist"], inst["service"], inst["ready"], inst["due"], inst["demand"]
    errors = []

    visited = [c for r in routes for c in r]
    if sorted(visited) != list(range(1, inst["n_customers"] + 1)):
        errors.append("each customer must be visited exactly once")
    if len(routes) > inst["n_vehicles"]:
        errors.append("too many vehicles")

    total = 0.0
    for k, r in enumerate(routes):
        if q[r].sum() > inst["capacity"] + 1e-9:
            errors.append(f"route {k}: capacity exceeded")
        t, prev = e[0], 0
        for c in r + [0]:                                  # simulate the route, back to the depot
            t = max(t + s[prev] + d[prev, c], e[c])        # wait if early
            if t > l[c] + 1e-9:
                errors.append(f"route {k}: late at node {c}")
                break
            total += d[prev, c]
            prev = c
    return total, errors

### Stages 3 and 4: the ALNS main loop

All parameters are gathered in `DEFAULT_PARAMS` (see the parameter table of the Solution section) and can be overridden in a call, e.g. `alns(inst, iterations=1000, seed=2)`.

In [7]:
DEFAULT_PARAMS = dict(
    iterations=2000, time_limit=None, seed=0,
    q_min_frac=0.05, q_max_frac=0.30, q_max_abs=40,
    worst_p=3, shaw_p=6, noise=0.025,
    vehicle_cost=0.0, unassigned_penalty=1e4,
    sigma1=33, sigma2=9, sigma3=13, reaction=0.1, segment=100,
    start_worse=0.05, start_accept=0.5, end_temp_frac=1e-4,
)


def alns(inst, **overrides):
    params = {**DEFAULT_PARAMS, **overrides}
    rng = np.random.default_rng(params["seed"])            # seeded -> reproducible runs
    t0 = time.perf_counter()

    # ---- Step 3: initial solution = repair an empty solution ----
    empty = Solution([], range(1, inst["n_customers"] + 1))
    current = regret2_repair(empty, inst, rng, params)
    f_current = current.objective(inst, params)
    best, f_best = current.copy(), f_current

    # ---- Step 7: simulated annealing temperature ----
    T = -params["start_worse"] * f_current / np.log(params["start_accept"])
    cooling = params["end_temp_frac"] ** (1.0 / params["iterations"])   # reach T0 * end_temp_frac at the end

    # ---- Step 6: adaptive weights, scores and usage counters ----
    nD, nR = len(DESTROY), len(REPAIR)
    wD, wR = np.ones(nD), np.ones(nR)
    scoreD, scoreR = np.zeros(nD), np.zeros(nR)
    usedD, usedR = np.zeros(nD), np.zeros(nR)
    history = {"current": [], "best": [], "wD": [wD.copy()], "wR": [wR.copy()], "initial": f_current}

    for it in range(params["iterations"]):
        if params["time_limit"] and time.perf_counter() - t0 > params["time_limit"]:
            break                                           # Step 8: optional time limit

        # Roulette-wheel selection of one destroy and one repair operator
        i = rng.choice(nD, p=wD / wD.sum())
        j = rng.choice(nR, p=wR / wR.sum())

        # Destroy then repair
        candidate = REPAIR[j](DESTROY[i](current, inst, rng, params), inst, rng, params)
        f_cand = candidate.objective(inst, params)

        # Acceptance + score of the operator pair
        score = 0
        if f_cand < f_best - 1e-9:                          # new global best
            best, f_best = candidate.copy(), f_cand
            current, f_current = candidate, f_cand
            score = params["sigma1"]
        elif f_cand < f_current - 1e-9:                     # improves the current solution
            current, f_current = candidate, f_cand
            score = params["sigma2"]
        elif rng.random() < np.exp(-(f_cand - f_current) / T):   # worse but accepted (SA rule)
            current, f_current = candidate, f_cand
            score = params["sigma3"]

        scoreD[i] += score; scoreR[j] += score
        usedD[i] += 1;      usedR[j] += 1
        T *= cooling
        history["current"].append(f_current)
        history["best"].append(f_best)

        # End of a segment: update the weights  w <- (1 - r) w + r * score / uses
        if (it + 1) % params["segment"] == 0:
            r = params["reaction"]
            wD = np.where(usedD > 0, (1 - r) * wD + r * scoreD / np.maximum(usedD, 1), wD)
            wR = np.where(usedR > 0, (1 - r) * wR + r * scoreR / np.maximum(usedR, 1), wR)
            wD, wR = np.maximum(wD, 0.05), np.maximum(wR, 0.05)   # never let an operator disappear completely
            scoreD[:] = scoreR[:] = usedD[:] = usedR[:] = 0
            history["wD"].append(wD.copy())
            history["wR"].append(wR.copy())

    history["runtime"] = time.perf_counter() - t0
    return best, history

## Test

In [9]:
#first try 
inst = load_instance(f"{DATA_DIR}/C1_2_1.200.20.vrptw")
alns_solution, alns_history = alns(inst, iterations=2000, seed=0, vehicle_cost=0.0, unassigned_penalty=1e4)
print("ALNS solution: distance =", alns_solution.distance(inst), ", #vehicles =", len(alns_solution.routes))


ALNS solution: distance = 2704.5677663304737 , #vehicles = 20


# **Experimental Study**

